# EDA — AI Governance in Brazilian Universities

**Projeto:** `ai-governance-brazilian-universities`  
**Dados:** `data/raw/levantamento_ia_universidades_brasileiras.csv` (27 registros)  
**PDFs:** `data/processed/documentos/` (20 arquivos)  

---

Este notebook está organizado em três partes:

1. **Análise exploratória dos metadados** — distribuição por ano, região, categoria, tipo de documento e status  
2. **Linha do tempo** — marcos históricos sobrepostos à curva de produção normativa  
3. **TF-IDF e classificação** — separar documentos *operacionais* (prescrevem comportamentos concretos) de *genéricos* (ficam no plano dos princípios)

---

In [ ]:
# ── Dependências ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import pdfplumber

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT       = Path("..")
CSV_PATH   = ROOT / "data" / "raw" / "levantamento_ia_universidades_brasileiras.csv"
PDF_DIR    = ROOT / "data" / "processed" / "documentos"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Estilo global ─────────────────────────────────────────────────────────────
PALETTE_CAT  = "#2D5FA8"   # azul institucional — cor principal
PALETTE_EMPH = "#C0392B"   # vermelho — destaques
GREY         = "#AAAAAA"

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi":        150,
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.spines.left":  False,
    "axes.grid":         True,
    "grid.color":        "#E5E5E5",
    "grid.linewidth":    0.6,
    "font.family":       "DejaVu Sans",
})

print("Ambiente pronto.")

---
## Parte 1 — Análise exploratória dos metadados

### 1.1  Carga e inspeção inicial

In [ ]:
df = pd.read_csv(CSV_PATH)

# Normalização mínima
df.columns = df.columns.str.strip()
df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")

# Flag: tem PDF baixado?
df["tem_pdf"] = df["arquivo"].notna() & (df["arquivo"].str.strip() != "")

print(f"Registros: {len(df)}")
print(f"PDFs baixados: {df['tem_pdf'].sum()} de {len(df)}")
print()
df.info()
df.head()

### 1.2  Distribuição por status

In [ ]:
status_counts = df["status"].value_counts()

STATUS_COLORS = {
    "publicado":          "#2D5FA8",
    "em andamento":       "#E67E22",
    "pesquisa interna":   "#8E44AD",
    "sem documento formal": GREY,
}
cores = [STATUS_COLORS.get(s, GREY) for s in status_counts.index]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(status_counts.index, status_counts.values, color=cores, height=0.55)
for bar, val in zip(bars, status_counts.values):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=11, fontweight="bold")

ax.set_xlabel("Registros", labelpad=8)
ax.set_title("Status dos documentos", fontsize=13, fontweight="bold", pad=12)
ax.set_xlim(0, status_counts.max() + 2)
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_status.png", bbox_inches="tight")
plt.show()

### 1.3  Distribuição por ano

In [ ]:
ano_counts = df.groupby("ano").size().reset_index(name="n")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(ano_counts["ano"].astype(str), ano_counts["n"],
       color=PALETTE_CAT, width=0.55, edgecolor="white")

for _, row in ano_counts.iterrows():
    ax.text(str(row["ano"]), row["n"] + 0.15, str(row["n"]),
            ha="center", fontsize=11, fontweight="bold")

ax.set_ylabel("Documentos", labelpad=8)
ax.set_title("Documentos por ano", fontsize=13, fontweight="bold", pad=12)
ax.set_ylim(0, ano_counts["n"].max() + 2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_por_ano.png", bbox_inches="tight")
plt.show()

### 1.4  Distribuição por região

In [ ]:
REGIAO_ORDER = ["Sudeste", "Nordeste", "Sul", "Centro-Oeste", "Norte", "Nacional"]
regiao_counts = (df["regiao"]
                 .value_counts()
                 .reindex(REGIAO_ORDER)
                 .dropna()
                 .reset_index())
regiao_counts.columns = ["regiao", "n"]

REGIAO_COLORS = {
    "Sudeste":     "#2D5FA8",
    "Nordeste":    "#1A8C6A",
    "Sul":         "#6E3D9E",
    "Centro-Oeste":"#D4820A",
    "Nacional":    GREY,
}
cores = [REGIAO_COLORS.get(r, GREY) for r in regiao_counts["regiao"]]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(regiao_counts["regiao"], regiao_counts["n"],
               color=cores, height=0.55)
for bar, val in zip(bars, regiao_counts["n"]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=11, fontweight="bold")

ax.set_xlabel("Registros", labelpad=8)
ax.set_title("Documentos por região", fontsize=13, fontweight="bold", pad=12)
ax.set_xlim(0, regiao_counts["n"].max() + 2)
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_por_regiao.png", bbox_inches="tight")
plt.show()

### 1.5  Distribuição por categoria institucional

In [ ]:
cat_counts = df["categoria"].value_counts().reset_index()
cat_counts.columns = ["categoria", "n"]

CAT_COLORS = {
    "Federal":             "#2D5FA8",
    "Privada":             "#C0392B",
    "Estadual":            "#1A8C6A",
    "Órgão Federal":       "#D4820A",
    "Associação Nacional": GREY,
}
cores = [CAT_COLORS.get(c, GREY) for c in cat_counts["categoria"]]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(cat_counts["categoria"], cat_counts["n"],
               color=cores, height=0.55)
for bar, val in zip(bars, cat_counts["n"]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=11, fontweight="bold")

ax.set_xlabel("Registros", labelpad=8)
ax.set_title("Documentos por categoria institucional", fontsize=13, fontweight="bold", pad=12)
ax.set_xlim(0, cat_counts["n"].max() + 2)
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_por_categoria.png", bbox_inches="tight")
plt.show()

### 1.6  Tipos de documento

In [ ]:
# Agrupa tipos similares para legibilidade
TIPO_MAP = {
    "Resolução":               "Resolução",
    "Guia":                    "Guia",
    "Diretriz normativa":      "Diretriz",
    "Diretrizes + Recomendações": "Diretriz",
    "Diretrizes":              "Diretriz",
    "Diretriz setorial":       "Diretriz",
    "Portaria":                "Portaria",
    "Orientações":             "Orientação",
    "Orientação setorial":     "Orientação",
    "Recomendações":           "Recomendações",
    "Recomendações (revisão)": "Recomendações",
    "Manual / dossiê":         "Manual",
    "Instrução normativa":     "Instrução normativa",
    "Referencial":             "Referencial",
    "Deliberação":             "Deliberação",
    "Política + Plano (consulta pública)": "Política / Plano",
    "Processo de normatização":"Processo (em andamento)",
    "Sem documento formal":    "Sem documento",
    "Pesquisa interna":        "Pesquisa interna",
}

df["tipo_agrupado"] = df["tipo_documento"].map(TIPO_MAP).fillna(df["tipo_documento"])
tipo_counts = df["tipo_agrupado"].value_counts().reset_index()
tipo_counts.columns = ["tipo", "n"]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.barh(tipo_counts["tipo"], tipo_counts["n"],
               color=PALETTE_CAT, height=0.6)
for bar, val in zip(bars, tipo_counts["n"]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=10, fontweight="bold")

ax.set_xlabel("Registros", labelpad=8)
ax.set_title("Tipos de documento (agrupados)", fontsize=13, fontweight="bold", pad=12)
ax.set_xlim(0, tipo_counts["n"].max() + 1.5)
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_tipos_doc.png", bbox_inches="tight")
plt.show()

### 1.7  Heatmap: ano × região

In [ ]:
pivot = (df[df["regiao"] != "Nacional"]
         .groupby(["ano", "regiao"])
         .size()
         .unstack(fill_value=0))

# Garante colunas na ordem geográfica
cols = [c for c in REGIAO_ORDER if c in pivot.columns and c != "Nacional"]
pivot = pivot[cols]

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(
    pivot,
    annot=True, fmt="d", cmap="Blues",
    linewidths=0.5, linecolor="#E0E0E0",
    cbar_kws={"label": "Documentos", "shrink": 0.8},
    ax=ax,
)
ax.set_title("Documentos publicados: ano × região",
             fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Ano", labelpad=8)
ax.set_xlabel("Região", labelpad=8)
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_heatmap_ano_regiao.png", bbox_inches="tight")
plt.show()

---
## Parte 2 — Linha do tempo com marcos históricos

In [ ]:
MARCOS = [
    (2022.92, "ChatGPT\n(nov. 2022)",          0.60),
    (2023.08, "Llama 1\n(fev. 2023)",          0.40),
    (2023.17, "GPT-4\n(mar. 2023)",            0.25),
    (2023.25, "Claude 1\n(mar. 2023)",         0.02),
    (2023.75, "Gemini\n(dez. 2023)",           0.55),
    (2024.58, "EU AI Act\n(ago. 2024)",        0.10),
    (2024.92, "PL 2338/2023\naprov. Senado",   0.50),
    (2025.17, "PL 2338/2023\nna Câmara",       0.25),
]

publi = df[df["status"] == "publicado"].copy()
ano_series = publi["ano"].dropna().astype(int).sort_values()
anos_unicos = sorted(ano_series.unique())
contagens   = [int((ano_series == a).sum()) for a in anos_unicos]
cumulativo  = list(np.cumsum(contagens))

fig, ax = plt.subplots(figsize=(12, 5.5))

ax.fill_between(anos_unicos, cumulativo, alpha=0.10, color=PALETTE_CAT)
ax.plot(anos_unicos, cumulativo, "o-", color=PALETTE_CAT,
        linewidth=2, markersize=7, zorder=3)

for ano_dec, label, lado in MARCOS:
    yval = max(cumulativo) * lado
    ax.axvline(ano_dec, color="#C0392B", linewidth=0.8, linestyle="--", alpha=0.6, zorder=1)
    ha = "left"
    offset = 0.04
    if label.startswith("Claude"):
        ha = "right"
        offset = -0.04
    ax.text(ano_dec + offset, yval, label,
            fontsize=8, color="#C0392B", va="center", ha=ha,
            linespacing=1.4,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75))

ax.set_xlabel("Ano", labelpad=8)
ax.set_ylabel("Documentos acumulados", labelpad=8)
ax.set_title("Produção normativa sobre IA em universidades brasileiras\ncom marcos históricos",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xticks(range(min(anos_unicos), max(anos_unicos) + 1))
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

handle_linha = mpatches.Patch(color=PALETTE_CAT, label="Acumulado publicados")
handle_marco = mpatches.Patch(color="#C0392B", alpha=0.6, label="Marco histórico")
ax.legend(handles=[handle_linha, handle_marco], loc="upper left", fontsize=9, framealpha=0.9)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_timeline.png", bbox_inches="tight")
plt.show()

---
## Parte 3 — TF-IDF e classificação operacional × genérico

**Critério de classificação**

| Classe | Definição operacional |
|---|---|
| **Operacional** | O documento contém pelo menos uma prescrição concreta: verbos no imperativo ("deve", "é vedado", "é obrigatório"), referências a punições ou sanções, ou vocabulário de controle e monitoramento. |
| **Genérico** | O documento permanece no plano dos princípios: "transparência", "responsabilidade", "ética" sem ligação a condutas específicas. |

### 3.1  Extração de texto dos PDFs

In [ ]:
def extrair_texto_pdf(caminho: Path, max_paginas: int = 30) -> str:
    """Extrai texto de um PDF com pdfplumber.
    Limita a max_paginas para evitar documentos extensos dominarem o corpus.
    """
    try:
        with pdfplumber.open(caminho) as pdf:
            paginas = pdf.pages[:max_paginas]
            texto = " ".join(
                (p.extract_text() or "") for p in paginas
            )
        return texto.strip()
    except Exception as e:
        print(f"  ERRO em {caminho.name}: {e}")
        return ""


# Filtra apenas registros com arquivo PDF listado
df_pdfs = df[df["tem_pdf"]].copy().reset_index(drop=True)

print(f"PDFs no levantamento: {len(df_pdfs)}")
print("Extraindo texto...")

textos = []
for _, row in df_pdfs.iterrows():
    caminho = PDF_DIR / row["arquivo"]
    if caminho.exists():
        texto = extrair_texto_pdf(caminho)
        textos.append(texto)
        status_txt = f"{len(texto):>7,} chars"
    else:
        textos.append("")
        status_txt = "ARQUIVO NÃO ENCONTRADO"
    print(f"  {row['arquivo']:<55} {status_txt}")

df_pdfs["texto"] = textos
df_pdfs["n_chars"] = df_pdfs["texto"].str.len()

validos = (df_pdfs["n_chars"] > 100).sum()
print(f"\nDocumentos com texto extraído com sucesso: {validos} de {len(df_pdfs)}")

### 3.2  Vocabulário e TF-IDF

In [ ]:
# ── Stopwords ─────────────────────────────────────────────────────────────────
# Mistura sklearn + manual (português)
STOPWORDS_PT = [
    "de","do","da","dos","das","em","no","na","nos","nas",
    "o","a","os","as","um","uma","e","é","que","com","por",
    "para","ou","se","ao","aos","às","pelo","pela","pelos","pelas",
    "entre","sobre","sua","seu","suas","seus","como","mais",
    "também","ser","ter","não","pode","ser","são","foi",
    "foram","está","este","essa","esse","esta","neste","nesta",
    "nesse","nessa","nos","nas","artigo","art","parágrafo",
    "inciso","alínea","§","ii","iii","iv","vi","vii","viii",
    "resolução","portaria","instrução","normativa",
    "universidade","federal","estadual","municipal",
    # adicionados
    "aos","tem","havia","há","seu","sua","assim","quando",
    "onde","qual","quais","todo","toda","todos","todas","são","sao",
    "outro","outra","outros","outras","mesmo","mesma","tambem",
]

corpus_valido = df_pdfs[df_pdfs["n_chars"] > 100].reset_index(drop=True)
print(f"Corpus para TF-IDF: {len(corpus_valido)} documentos")

vectorizer = TfidfVectorizer(
    max_features=500,
    min_df=1,
    ngram_range=(1, 2),      # unigramas + bigramas
    stop_words=STOPWORDS_PT,
    strip_accents="unicode",
    lowercase=True,
    sublinear_tf=True,       # log(1+tf) — suaviza documentos longos
)

X = vectorizer.fit_transform(corpus_valido["texto"])
feature_names = vectorizer.get_feature_names_out()

print(f"Vocabulário (top features): {len(feature_names)} termos")
print(f"Matriz TF-IDF: {X.shape}")

### 3.3  Classificação heurística: operacional × genérico

In [ ]:
# ── Dicionários de prescrição e princípios ────────────────────────────────────
TERMOS_OPERACIONAIS = [
    # Verbos de obrigação
    "deve", "devem", "deverá", "deverão", "é obrigatório", "é vedado",
    "é proibido", "fica proibido", "fica vedado", "é permitido",
    "é necessário", "é exigido",
    # Penalidades e controle
    "sanção", "penalidade", "advertência", "suspensão", "punição",
    "reprovação", "anulação", "comissão", "comitê",
    # Monitoramento
    "monitoramento", "auditoria", "fiscalização", "relatório",
    "revisão periódica", "avaliação",
    # Procedimentos
    "declarar", "declaração", "formulário", "protocolo",
    "credenciamento", "licença",
]

TERMOS_GENERICOS = [
    "ética", "ético", "responsabilidade", "transparência",
    "princípio", "valor", "compromisso", "diálogo",
    "reflexão", "conscientização", "sensibilização",
    "recomenda", "sugere", "incentiva", "estimula",
    "boas práticas", "orientação", "orientações",
]


def score_operacional(texto: str) -> float:
    """Score entre 0 e 1: proporção (ponderada) de ocorrências operacionais
    vs. genéricas no texto.
    """
    txt = texto.lower()
    op  = sum(txt.count(t) for t in TERMOS_OPERACIONAIS)
    gen = sum(txt.count(t) for t in TERMOS_GENERICOS)
    total = op + gen
    if total == 0:
        return 0.0
    return round(op / total, 4)


corpus_valido = corpus_valido.copy()
corpus_valido["score_op"]  = corpus_valido["texto"].apply(score_operacional)

# Threshold: >= 0.40 → operacional
THRESHOLD = 0.40
corpus_valido["classe"] = corpus_valido["score_op"].apply(
    lambda s: "Operacional" if s >= THRESHOLD else "Genérico"
)

# Relatório
print(f"Threshold: {THRESHOLD:.0%}\n")
display_cols = ["instituicao", "ano", "tipo_agrupado", "score_op", "classe"]
print(corpus_valido[display_cols].sort_values("score_op", ascending=False).to_string(index=False))
print()
print(corpus_valido["classe"].value_counts().to_string())

### 3.4  Visualização dos scores

In [ ]:
# Ordena por score
plot_df = corpus_valido.sort_values("score_op", ascending=True).reset_index(drop=True)
plot_df["label"] = plot_df["instituicao"] + " (" + plot_df["ano"].astype(str) + ")"

cores = ["#C0392B" if c == "Operacional" else "#2D5FA8" for c in plot_df["classe"]]

fig, ax = plt.subplots(figsize=(8, max(4, len(plot_df) * 0.42)))
bars = ax.barh(plot_df["label"], plot_df["score_op"],
               color=cores, height=0.6)
ax.axvline(THRESHOLD, color="#888", linewidth=1, linestyle="--",
           label=f"Threshold ({THRESHOLD:.0%})")

for bar, val in zip(bars, plot_df["score_op"]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=8.5)

handle_op  = mpatches.Patch(color="#C0392B", label="Operacional")
handle_gen = mpatches.Patch(color="#2D5FA8", label="Genérico")
handle_thr = mpatches.Patch(color="#888",    label=f"Threshold ({THRESHOLD:.0%})")
ax.legend(handles=[handle_op, handle_gen, handle_thr], fontsize=9, loc="lower right")

ax.set_xlabel("Score operacional (proporção termos prescritivos)", labelpad=8)
ax.set_title("Classificação: documentos operacionais × genéricos",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlim(0, 1.05)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "08_classificacao_operacional.png", bbox_inches="tight")
plt.show()

### 3.5  Top termos TF-IDF por classe

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

TOP_N = 20

def top_termos_classe(df_classe, classe_nome, cor, ax):
    textos = df_classe[df_classe["classe"] == classe_nome]["texto"]
    if len(textos) == 0:
        ax.set_title(f"{classe_nome} — sem documentos")
        return

    vec = TfidfVectorizer(
        max_features=200,
        ngram_range=(1, 2),
        stop_words=STOPWORDS_PT,
        strip_accents="unicode",
        lowercase=True,
        sublinear_tf=True,
    )
    X_c = vec.fit_transform(textos)
    scores = np.asarray(X_c.mean(axis=0)).flatten()
    termos = vec.get_feature_names_out()
    idx = scores.argsort()[-TOP_N:][::-1]

    ax.barh(
        [termos[i] for i in idx][::-1],
        [scores[i] for i in idx][::-1],
        color=cor, height=0.65,
    )
    ax.set_title(f"Top {TOP_N} termos — {classe_nome} ({len(textos)} docs)",
                 fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel("TF-IDF médio", fontsize=9)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
top_termos_classe(corpus_valido, "Operacional", "#C0392B", ax1)
top_termos_classe(corpus_valido, "Genérico",    "#2D5FA8", ax2)
fig.suptitle("Termos mais relevantes por classe (TF-IDF médio)",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "09_tfidf_por_classe.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Dicionários refinados (destinatário + conduta) ────────────────────────────
TERMOS_OP_REFINADO = [
    "o aluno deve", "o docente deve", "o estudante deve",
    "é obrigatório", "é vedado", "é proibido", "fica proibido",
    "não é permitido", "não pode ser utilizado",
    "declarar", "declaração", "indicar o uso", "informar o uso",
    "citar a ferramenta", "referenciar", "identificar",
    "reprovação", "anulação", "sanção", "advertência",
    "infração", "plágio", "desonestidade acadêmica",
    "submeter", "protocolo", "formulário", "aprovação prévia",
]

TERMOS_GEN_REFINADO = [
    "ética", "ético", "responsabilidade", "transparência",
    "princípio", "valor", "compromisso", "equidade",
    "reflexão", "conscientização", "sensibilização",
    "recomenda-se", "sugere-se", "incentiva-se",
    "boas práticas", "uso responsável", "uso ético",
    "sociedade", "humanidade", "bem comum",
]


def score_operacional_refinado(texto: str) -> float:
    txt = texto.lower()
    op  = sum(txt.count(t) for t in TERMOS_OP_REFINADO)
    gen = sum(txt.count(t) for t in TERMOS_GEN_REFINADO)
    total = op + gen
    if total == 0:
        return 0.0
    return round(op / total, 4)


corpus_valido["score_op_ref"] = corpus_valido["texto"].apply(score_operacional_refinado)
corpus_valido["classe_ref"] = corpus_valido["score_op_ref"].apply(
    lambda s: "Operacional" if s >= THRESHOLD else "Genérico"
)

# Comparação lado a lado
comparacao = corpus_valido[[
    "instituicao", "ano", "score_op", "classe", "score_op_ref", "classe_ref"
]].sort_values("score_op_ref", ascending=False)

# Marca divergências
comparacao["diverge"] = comparacao["classe"] != comparacao["classe_ref"]

print(comparacao.to_string(index=False))
print(f"\nDivergências entre as duas abordagens: {comparacao['diverge'].sum()}")

In [ ]:
def top_termos_classe_refinado(df_classe, classe_nome, cor, ax):
    textos = df_classe[df_classe["classe_ref"] == classe_nome]["texto"]
    if len(textos) == 0:
        ax.set_title(f"{classe_nome} — sem documentos")
        return

    vec = TfidfVectorizer(
        max_features=200,
        ngram_range=(1, 2),
        stop_words=STOPWORDS_PT,
        strip_accents="unicode",
        lowercase=True,
        sublinear_tf=True,
    )
    X_c = vec.fit_transform(textos)
    scores = np.asarray(X_c.mean(axis=0)).flatten()
    termos = vec.get_feature_names_out()
    idx = scores.argsort()[-TOP_N:][::-1]

    ax.barh(
        [termos[i] for i in idx][::-1],
        [scores[i] for i in idx][::-1],
        color=cor, height=0.65,
    )
    ax.set_title(f"Top {TOP_N} termos — {classe_nome} ({len(textos)} docs)\n(classificação refinada)",
                 fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel("TF-IDF médio", fontsize=9)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
top_termos_classe_refinado(corpus_valido, "Operacional", "#C0392B", ax1)
top_termos_classe_refinado(corpus_valido, "Genérico",    "#2D5FA8", ax2)
fig.suptitle("Termos mais relevantes por classe — classificação refinada (TF-IDF médio)",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "10_tfidf_por_classe_refinado.png", bbox_inches="tight")
plt.show()

### 3.5a  Termos exclusivos por classe (classificação refinada)

O TF-IDF médio por classe (célula 3.5) mostrou pouca diferença entre os grupos
devido à homogeneidade do corpus — todos os documentos compartilham o mesmo
vocabulário acadêmico-jurídico sobre IA.

Esta célula adota uma abordagem alternativa: em vez de ponderar por raridade no
corpus, calcula a **diferença relativa de frequência** entre os dois grupos.
Isso revela os termos que aparecem proporcionalmente mais em um grupo do que no
outro — ou seja, o vocabulário que de fato distingue os documentos operacionais
dos genéricos neste corpus.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def termos_exclusivos(df, classe_alvo, classe_col, texto_col, top_n=20):
    txt_alvo   = " ".join(df[df[classe_col] == classe_alvo][texto_col])
    txt_outro  = " ".join(df[df[classe_col] != classe_alvo][texto_col])

    vec = CountVectorizer(ngram_range=(1,2), stop_words=STOPWORDS_PT,
                          strip_accents="unicode", lowercase=True)
    vec.fit([txt_alvo, txt_outro])
    nomes = vec.get_feature_names_out()

    counts_alvo  = vec.transform([txt_alvo]).toarray()[0]
    counts_outro = vec.transform([txt_outro]).toarray()[0]

    # Normaliza pelo tamanho do texto
    freq_alvo  = counts_alvo  / (counts_alvo.sum()  + 1)
    freq_outro = counts_outro / (counts_outro.sum() + 1)

    # Diferença relativa
    diff = freq_alvo - freq_outro
    idx  = diff.argsort()[-top_n:][::-1]
    return [(nomes[i], round(diff[i]*1000, 3)) for i in idx]

print("Termos mais exclusivos dos Operacionais refinados:")
for termo, score in termos_exclusivos(corpus_valido, "Operacional", "classe_ref", "texto"):
    print(f"  {termo:<35} {score}")

print("\nTermos mais exclusivos dos Genéricos refinados:")
for termo, score in termos_exclusivos(corpus_valido, "Genérico", "classe_ref", "texto"):
    print(f"  {termo:<35} {score}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

op_termos  = termos_exclusivos(corpus_valido, "Operacional", "classe_ref", "texto")
gen_termos = termos_exclusivos(corpus_valido, "Genérico",    "classe_ref", "texto")

for ax, termos, cor, titulo in [
    (ax1, op_termos,  "#C0392B", "Operacionais refinados"),
    (ax2, gen_termos, "#2D5FA8", "Genéricos refinados"),
]:
    labels = [t[0] for t in termos][::-1]
    values = [t[1] for t in termos][::-1]
    ax.barh(labels, values, color=cor, height=0.65)
    ax.set_title(f"Termos exclusivos — {titulo}",
                 fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel("Diferença relativa de frequência (×1000)", fontsize=9)

fig.suptitle("Vocabulário distintivo por classe (classificação refinada)",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "11_termos_exclusivos_refinado.png", bbox_inches="tight")
plt.show()

### 3.6  Exporta tabela de resultados

In [ ]:
# Garante que as colunas refinadas existem
if "classe_ref" in corpus_valido.columns and "diverge" not in corpus_valido.columns:
    corpus_valido["diverge"] = corpus_valido["classe"] != corpus_valido["classe_ref"]

# CSV 1 — classificação original
resultado_original = corpus_valido[[
    "instituicao", "estado", "regiao", "categoria",
    "ano", "tipo_agrupado", "titulo_documento",
    "status", "arquivo", "score_op", "classe",
]].copy()

saida_original = OUTPUT_DIR / "classificacao_original.csv"
resultado_original.to_csv(saida_original, index=False, encoding="utf-8-sig")
print(f"Salvo: {saida_original}")

# CSV 2 — classificação refinada + comparação
resultado_refinado = corpus_valido[[
    "instituicao", "estado", "regiao", "categoria",
    "ano", "tipo_agrupado", "titulo_documento",
    "status", "arquivo",
    "score_op", "classe",
    "score_op_ref", "classe_ref", "diverge",
]].copy()

saida_refinada = OUTPUT_DIR / "classificacao_refinada.csv"
resultado_refinado.to_csv(saida_refinada, index=False, encoding="utf-8-sig")
print(f"Salvo: {saida_refinada}")

resultado_refinado.sort_values("score_op_ref", ascending=False)

---
## Sumário

| Seção | Output |
|---|---|
| 1.2 Status | `outputs/01_status.png` |
| 1.3 Por ano | `outputs/02_por_ano.png` |
| 1.4 Por região | `outputs/03_por_regiao.png` |
| 1.5 Por categoria | `outputs/04_por_categoria.png` |
| 1.6 Tipos | `outputs/05_tipos_doc.png` |
| 1.7 Heatmap | `outputs/06_heatmap_ano_regiao.png` |
| 2 Linha do tempo | `outputs/07_timeline.png` |
| 3.4 Classificação | `outputs/08_classificacao_operacional.png` |
| 3.5 TF-IDF por classe | `outputs/09_tfidf_por_classe.png` |
| 3.6 Tabela | `outputs/classificacao_operacional_vs_generico.csv` |

**Próximos passos sugeridos:**
- Validar manualmente a classificação operacional × genérico em uma amostra de 5–10 documentos
- Ajustar `THRESHOLD` e/ou dicionários após revisão qualitativa
- Substituir a heurística de dicionário por um classificador supervisionado (LogReg/SVM) quando houver rótulos manuais suficientes
- Adicionar análise de co-ocorrência de termos (`networkx`) para mapear campos semânticos dominantes